# Исторический статистический анализ ОКС

Исходный исследовательский код сохранён с изменением путей и удалением outputs.
Внутригоспитальный исход здесь кодируется как 0 — смерть, 1 — выжил.
Глобальная предобработка и предварительная балансировка ограничивают оценку ML.
Для основного ML-протокола используйте machine_learning/notebooks/01→02→03.
Фигуры с искусственно подогнанными средними/CV являются иллюстрациями,
их нельзя выдавать за фактические повторные эксперименты Монте-Карло.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "studies/acs/configs/layout.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри клонированного acs-research-projects")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from acs_research.paths import study_paths
P = study_paths(PROJECT_ROOT)
STUDY = P.study
OUT = P.statistics
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in
            ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
print(platform.python_version(), "seed", BASE_SEED, VERSIONS)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def figsave(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight")
    plt.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr, fisher_exact
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE
import statsmodels.api as sm
import shap
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

In [ ]:
file_name = P.inputs / "curated_clinical.csv"
df = pd.read_csv(file_name)

# Подготовка данных

In [ ]:
df = df.drop(columns=['Статус', 'Причина смерти', 'Дата смерти', 'Иифаркт миокарда за время наблюдения',\
                     'Инсульт за время наблюдения', 'ЧКВ или КШ за время наблюдения', 'Выполнение рекомендаций',\
                     'Статус.1', 'Причина смерти.1', 'Дата смерти.1', 'Иифаркт миокарда за время наблюдения.1',\
                     'Инсульт за время наблюдения.1', 'ЧКВ или КШ за время наблюдения.1', 'Выполнение рекомендаций.1', 'Число койко-дней'], errors='ignore')

In [ ]:
df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

def clean_numeric(value):
    if isinstance(value, str):
        value = value.replace(',', '.')
        if 'N' in value or value.strip() == '':
            return np.nan
    try:
        return float(value)
    except:
        return np.nan

cols_to_clean = [
    'Возраст в годах',
    'Максимальный уровень ТАГ в ммоль/л',
    'СРБ В мг/л N - не оценивалось',
    'Класс сердечной недостаточности по Killip',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось'
]


for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

In [ ]:
df_raw = df.copy()

In [ ]:
col_intervention = 'Интервенционное вмешательство 0 - не выполнено; 1 - стентирование; 2 - балонная ангиопластика без стентирования; 3 - только коронарография'

df_raw['Интервенционное вмешательство'] = df_raw[col_intervention].replace({
    1: 1,
    2: 1,
    0: 0,
    3: 0
})

In [ ]:
extreme_cols = [
    'ОНМК в анамнезе 0 - нет; 1 - есть',
    'Инфаркт миокарда в анамнезе 0 - нет; 1 - есть',
    'Коронарное шунтирование в анамнезе 0 - нет; 1 - есть',
    'ЧТКА со стентированием в анамнезе 0 - нет; 1 - есть',
]

df_raw['Экстремальный риск'] = (df[extreme_cols].sum(axis=1) > 0).astype(int)

In [ ]:
features_list =[
    'Возраст в годах',
    'Пол 1 - мужской; 0 - женский',
    'Заболевание 0-стенокардия; 1 - инфаркт миокарда',
    'Тип ОКС 0 - без подъема сегмента ST; 1 - с подъемом сегмента ST',
    'Максимальный уровень ТАГ в ммоль/л',
    'Метаболический синдром 0 - нет; 1 - есть',
    'Класс сердечной недостаточности по Killip',
    'ФВ менее, либо равно 45% 0 - нет; 1 - да',
    'СРБ В мг/л N - не оценивалось',
    'Хроническая болезнь почек 0 - нет; 1 - есть',
    'Догоспитальная ТЛТ 0 - нет; 1 - есть',
    'Фибрилляция предсердий 0 - нет; 1 - есть',
    'Интервенционное вмешательство',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось',
    'Сахарный диабет в анамнезе (известно до госпитализации) 0 - нет; 1 - есть',
    'Курение в анамнезе 0 - нет; 1 - есть',
    'Облитерирующий атеросклероз нижних конечностей в анамнезе 0 - нет; 1 - есть',
    'Экстремальный риск'
]
target_col = 'Исход госпитализации 1 - жив; 0 - умер'

In [ ]:
# col_killip = 'Класс сердечной недостаточности по Killip'

# cols_to_drop = [c for c in df_raw.columns if c.startswith('Killip_')]
# df_raw = df_raw.drop(columns=cols_to_drop)

# # 2. Применяем One-Hot Encoding
# killip_dummies = pd.get_dummies(df_raw[col_killip], prefix='Killip').astype(int)

# # 3. Присоединяем новые столбцы к df_raw
# df_raw = pd.concat([df_raw, killip_dummies], axis=1)

# # Сохраним названия новых колонок
# new_killip_features = killip_dummies.columns.tolist()

# # 4. Обновляем features_list
# # Важно: обновляем список только если в нем еще есть старое название
# if col_killip in features_list:
#     idx = features_list.index(col_killip)
#     # Заменяем один Killip на список новых колонок
#     features_list[idx:idx+1] = new_killip_features

# df_raw['Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось'].describe()

In [ ]:
cols_with_outliers = [
    'Максимальный уровень ТАГ в ммоль/л',
    'СРБ В мг/л N - не оценивалось',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось'
]

for col in cols_with_outliers:
    if col in df_raw.columns:
        # Вычисляем порог
        upper_limit = df_raw[col].quantile(0.975)
        
        # Считаем, сколько значений выше порога
        outliers_count = len(df_raw[df_raw[col] > upper_limit])
        
        # Применяем ограничение
        df_raw[col] = df_raw[col].clip(upper=upper_limit)

In [ ]:
# Подготавливаем датасет: удаляем строки, где целевая переменная не указана
df_ml = df_raw.dropna(subset=[target_col]).copy()

X = df_ml[features_list]
y = df_ml[[target_col]]

# Заполняем пропуски медианными значениями (стандартная практика для мед. данных)
X = X.fillna(X.median())

In [ ]:
cols_for_describe = [
    'Исход госпитализации 1 - жив; 0 - умер', 
    'Максимальный уровень ТАГ в ммоль/л',
    'Метаболический синдром 0 - нет; 1 - есть', 
    'Возраст в годах', 
    'ФВ менее, либо равно 45% 0 - нет; 1 - да',
    'Догоспитальная ТЛТ 0 - нет; 1 - есть', 
    'Экстремальный риск', 
    'СРБ В мг/л N - не оценивалось',
]

desc_table = df_raw[cols_for_describe].describe()
styled_table = desc_table.style.set_properties(**{
    'background-color': 'white',
    'color': 'black',
    'border-color': 'lightgrey',
    'border-style': 'solid',
    'border-width': '1px'
}).format("{:.2f}")

display(styled_table)

# Проверка связи ЭР и ур-ня ТАГ (ТАГ - некатегориальный)

In [ ]:
sns.histplot(y = df_raw['Экстремальный риск'], x = df_raw['Максимальный уровень ТАГ в ммоль/л'])
plt.show()

In [ ]:
# pearson_corr = df_raw.corr(method='pearson')
# pearson_corr

# Проверка связи ЭР и ур-ня ТАГ при наличии МС (ТАГ - некатегориальный)

In [ ]:
# 1. Подготовка названий колонок
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Экстремальный риск'

# 2. Настройка графика
plt.figure(figsize=(12, 7))

# Используем stripplot для визуализации распределения
# jitter=0.25 разносит точки, чтобы они не накладывались друг на друга
sns.stripplot(
    data=df_raw, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'red', 0: 'royalblue'}, # 1-Жив (синий), 0-Умер (красный)
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

# 3. Добавим клинический порог ТАГ (например, 1.7 ммоль/л) для наглядности
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

# 4. Оформление
plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от ЭР', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

# Настройка легенды
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Нет в группе ЭР (0)', 'Есть в группе ЭР (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
col_target_er = 'Экстремальный риск'

df_heat_er = df_raw.copy() # Используем исходные данные, как в вашем примере
df_heat_er['TAG_Category'] = np.where(df_heat_er[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

heatmap_data_er = df_heat_er.groupby(['TAG_Category', col_ms])[col_target_er].apply(
    lambda x: (x == 1).sum() / len(x) * 100
).unstack()

# 3. Подготовка подписей
er_annot = df_heat_er.groupby(['TAG_Category', col_ms])[col_target_er].apply(
    lambda x: f"{((x == 1).sum() / len(x) * 100):.1f}"
).unstack()

# 4. Визуализация
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data_er, annot=er_annot.values, fmt="", cmap='YlOrRd', cbar_kws={'label': '% пациентов в группе Экстр. риска'})
plt.title('Тепловая карта риска: группа Экстремального риска в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Метаболический синдром', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.xticks([0.5, 1.5], ['Без МС', 'С МС'])
plt.show()

# Проверка связи Исхода госпитализации и ур-ня ТАГ (ТАГ - некатегориальный)

In [ ]:
sns.histplot(y = df_raw['Исход госпитализации 1 - жив; 0 - умер'], x = df_raw['Максимальный уровень ТАГ в ммоль/л'])
plt.show()

# Проверка связи Исхода госпитализации и ур-ня ТАГ при наличии МС (ТАГ - некатегориальный)

In [ ]:
# 1. Подготовка названий колонок
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

# 2. Настройка графика
plt.figure(figsize=(12, 7))

# Используем stripplot для визуализации распределения
# jitter=0.25 разносит точки, чтобы они не накладывались друг на друга
sns.stripplot(
    data=df_raw, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'royalblue', 0: 'red'}, # 1-Жив (синий), 0-Умер (красный)
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

# 3. Добавим клинический порог ТАГ (например, 1.7 ммоль/л) для наглядности
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

# 4. Оформление
plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от Исхода', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

# Настройка легенды
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Умер (0)', 'Жив (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
# 1. Подготовка данных: создаем категории для ТАГ
df_heat_ishod = df_raw.copy()
df_heat_ishod['TAG_Category'] = np.where(df_heat_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

# 2. Создаем сводную таблицу (Pivot Table) для расчета процента смертей
# Нам нужно посчитать (количество 0) / (общее количество) * 100
heatmap_data_ishod = df_heat_ishod.groupby(['TAG_Category', col_ms])[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# Переименуем индексы и колонки для красоты
df_heat_ishod['TAG_Category'] = pd.Categorical(
    np.where(df_heat_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)'),
    categories=['Низкий (<1.7)', 'Высокий (>=1.7)'],  # явный порядок
    ordered=True
)

heatmap_data_ishod = df_heat_ishod.groupby(
    ['TAG_Category', col_ms], observed=True
)[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# Теперь index уже правильный — не нужно переназначать вручную
heatmap_data_ishod.columns = ['Без МС', 'С МС']
# index будет автоматически: ['Низкий (<1.7)', 'Высокий (>=1.7)']
# 3. Строим Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data_ishod, 
    annot=True,     # Включить цифры внутри ячеек
    fmt=".1f",      # Округлить до 1 знака после запятой
    cmap='YlOrRd',  # Цветовая гамма от желтого к красному (чем краснее, тем опаснее)
    cbar_kws={'label': '% Смертности в группе'}
)

plt.title('Тепловая карта риска для всей выборки: Процент смертности в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Состояние Метаболического синдрома', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.show()

# Проверка связи ФВ менее 45% и ур-ня ТАГ (ТАГ - некатегориальный)

In [ ]:
sns.histplot(y = df_raw['ФВ менее, либо равно 45% 0 - нет; 1 - да'], x = df_raw['Максимальный уровень ТАГ в ммоль/л'])
plt.show()

# Проверка связи ФВ менее 45% и ур-ня ТАГ при наличии МС (ТАГ - некатегориальный)

In [ ]:
# 1. Подготовка названий колонок
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'ФВ менее, либо равно 45% 0 - нет; 1 - да'

# 2. Настройка графика
plt.figure(figsize=(12, 7))

# Используем stripplot для визуализации распределения
# jitter=0.25 разносит точки, чтобы они не накладывались друг на друга
sns.stripplot(
    data=df_raw, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={0: 'royalblue', 1: 'red'}, # 1-Жив (синий), 0-Умер (красный)
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

# 3. Добавим клинический порог ТАГ (например, 1.7 ммоль/л) для наглядности
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

# 4. Оформление
plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от ФВ', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

# Настройка легенды
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['ФВ более 45% (1)', 'ФВ менее, либо равно 45% (0)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
# Названия колонок из вашего кода
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target_fv = 'ФВ менее, либо равно 45% 0 - нет; 1 - да'

# 1. Категоризация ТАГ
df_heat_fv = df_raw.copy()
df_heat_fv['TAG_Cat'] = np.where(df_heat_fv[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

# 2. Расчет % пациентов с ФВ <= 45% (цель == 1)
heatmap_data_fv = df_heat_fv.groupby(['TAG_Cat', col_ms])[col_target_fv].apply(
    lambda x: (x == 1).sum() / len(x) * 100
).unstack()

# 3. Подготовка подписей (Процент и кол-во)
fv_annot = df_heat_fv.groupby(['TAG_Cat', col_ms])[col_target_fv].apply(
    lambda x: f"{((x == 1).sum() / len(x) * 100):.1f}"
).unstack()

# 4. Визуализация
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data_fv, annot=fv_annot.values, fmt="", cmap='YlOrRd', cbar_kws={'label': '% пациентов с ФВ <= 45%'})
plt.title('Тепловая карта риска: Риск снижения ФВ менее 45% в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Метаболический синдром', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.xticks([0.5, 1.5], ['Без МС', 'С МС'])
plt.show()

# Таблица сопряженности для проверки зависимости категориальных факторов (Долгоспитальная ТЛТ и исход)

In [ ]:
col_ef = 'Догоспитальная ТЛТ 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

ct_1 = pd.crosstab(
    df_raw[col_ef], 
    df_raw[col_target], 
    margins=False
)

ct_1.index = ['ТЛТ не было (0)', 'ТЛТ было (1)']
ct_1.columns = ['Умерло (0)', 'Выжило (1)']

display(ct_1)

In [ ]:
odds_ratio_1, p_value_1 = fisher_exact(ct_1)

total_normal_ef_1 = ct_1.loc['ТЛТ не было (0)'].sum()
total_low_ef_1 = ct_1.loc['ТЛТ было (1)'].sum()

mortality_normal_1 = (ct_1.loc['ТЛТ не было (0)', 'Умерло (0)'] / total_normal_ef_1) * 100
mortality_low_1 = (ct_1.loc['ТЛТ было (1)', 'Умерло (0)'] / total_low_ef_1) * 100

print(f"Летальность в группе ТЛТ не было (0): {mortality_normal_1:.2f}%")
print(f"Летальность в группе ТЛТ было (1): {mortality_low_1:.2f}%")

print(f"P-value (Фишер): {p_value_1:.4f}")
print(f"Отношение шансов (Odds Ratio): {odds_ratio_1:.2f}")

In [ ]:
plt.figure(figsize=(6, 5))
rates_1 = pd.Series([mortality_normal_1, mortality_low_1], index=['ТЛТ не было (0)', 'ТЛТ было (1)'])
sns.barplot(x=rates_1.index, y=rates_1.values, palette='coolwarm')

plt.title('Сравнение летальности в зависимости от ТЛТ')
for i, v in enumerate(rates_1.values):
    plt.text(i, v, f"{v:.1f}%", ha='center', fontweight='bold')
plt.show()

# Таблица сопряженности для проверки зависимости категориальных факторов (Фибрилляция предсердий и исход)

In [ ]:
col_ef = 'Фибрилляция предсердий 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

ct_2 = pd.crosstab(
    df_raw[col_ef], 
    df_raw[col_target], 
    margins=False
)

ct_2.index = ['Фибрилляция нет (0)', 'Фибрилляция есть (1)']
ct_2.columns = ['Умерло (0)', 'Выжило (1)']

display(ct_2)

In [ ]:
odds_ratio_2, p_value_2 = fisher_exact(ct_2)

total_normal_ef_2 = ct_2.loc['Фибрилляция нет (0)'].sum()
total_low_ef_2 = ct_2.loc['Фибрилляция есть (1)'].sum()

mortality_normal_2 = (ct_2.loc['Фибрилляция нет (0)', 'Умерло (0)'] / total_normal_ef_2) * 100
mortality_low_2 = (ct_2.loc['Фибрилляция есть (1)', 'Умерло (0)'] / total_low_ef_2) * 100

print(f"Летальность в группе ТЛТ не было (0): {mortality_normal_2:.2f}%")
print(f"Летальность в группе ТЛТ было (1): {mortality_low_2:.2f}%")

print(f"P-value (Фишер): {p_value_2:.4f}")
print(f"Отношение шансов (Odds Ratio): {odds_ratio_2:.2f}")

In [ ]:
plt.figure(figsize=(6, 5))
rates_2 = pd.Series([mortality_normal_2, mortality_low_2], index=['Фибрилляция нет (0)', 'Фибрилляция есть (1)'])
sns.barplot(x=rates_2.index, y=rates_2.values, palette='coolwarm')

plt.title('Сравнение летальности в зависимости от фибрилляции')
for i, v in enumerate(rates_2.values):
    plt.text(i, v, f"{v:.1f}%", ha='center', fontweight='bold')
plt.show()

# Таблица сопряженности для категориальных факторов Killip и Исход госпитализации

In [ ]:
col_killip = 'Класс сердечной недостаточности по Killip'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

# 1. Построение расширенной таблицы сопряженности
ct_3 = pd.crosstab(df_raw[col_killip], df_raw[col_target])
ct_3.columns = ['Умерло (0)', 'Выжило (1)']

ct_3['Летальность (%)'] = (ct_3['Умерло (0)'] / ct_3.sum(axis=1)) * 100

display(ct_3)

In [ ]:
chi2, p, dof, expected = chi2_contingency(ct_3[['Умерло (0)', 'Выжило (1)']])

print(f"\nРезультат теста Хи-квадрат:")
print(f"P-value = {p:.4e}") # Используем научный формат, так как число будет очень маленьким

# 4. Расчет Отношения Шансов (OR) для каждого класса относительно Killip 1.0
# Шанс (odds) = умерло / выжило
base_odds = ct_3.loc[1.0, 'Умерло (0)'] / ct_3.loc[1.0, 'Выжило (1)']

print("\n--- ОТНОШЕНИЕ ШАНСОВ (ОШ) ОТНОСИТЕЛЬНО КЛАССА 1 ---")
for k_class in [2.0, 3.0, 4.0]:
    current_odds = ct_3.loc[k_class, 'Умерло (0)'] / ct_3.loc[k_class, 'Выжило (1)']
    or_val = current_odds / base_odds
    print(f"Класс {k_class} vs Класс 1.0: OR = {or_val:.2f}")

In [ ]:
# 5. Визуализация тренда летальности
plt.figure(figsize=(10, 6))
sns.barplot(x=ct_3.index, y=ct_3['Летальность (%)'], palette='Reds')
plt.title('Рост летальности в зависимости от тяжести по Killip', fontsize=14)
plt.xlabel('Класс Killip', fontsize=12)
plt.ylabel('Летальность (%)', fontsize=12)

# Добавляем подписи процентов над столбцами
for i, val in enumerate(ct_3['Летальность (%)']):
    plt.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold')

plt.show()

# Делаем ТАГ категориальным фактором (ТАГ > 1.7)

In [ ]:
df_tag_new = df_raw.copy()

col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

df_tag_new['ТАГ > 1.7'] = np.where(df_tag_new[col_tag] > 1.7, 1, 0)

# Таблица сопряженности для проверки зависимости категориальных факторов (ТАГ > 1.7 и исход)

In [ ]:
col_ef = 'ТАГ > 1.7'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

ct_4 = pd.crosstab(
    df_tag_new[col_ef], 
    df_tag_new[col_target], 
    margins=False
)

ct_4.index = ['ТАГ > 1.7', 'ТАГ <= 1.7']
ct_4.columns = ['Умерло (0)', 'Выжило (1)']

display(ct_4)

In [ ]:
odds_ratio_4, p_value_4 = fisher_exact(ct_4)

total_normal_ef_4 = ct_4.loc['ТАГ > 1.7'].sum()
total_low_ef_4 = ct_4.loc['ТАГ <= 1.7'].sum()

mortality_normal_4 = (ct_4.loc['ТАГ > 1.7', 'Умерло (0)'] / total_normal_ef_4) * 100
mortality_low_4 = (ct_4.loc['ТАГ <= 1.7', 'Умерло (0)'] / total_low_ef_4) * 100

print(f"Летальность в группе ТЛТ не было (0): {mortality_normal_4:.2f}%")
print(f"Летальность в группе ТЛТ было (1): {mortality_low_4:.2f}%")

print(f"P-value (Фишер): {p_value_4:.4f}")
print(f"Отношение шансов (Odds Ratio): {odds_ratio_4:.2f}")

In [ ]:
plt.figure(figsize=(6, 5))
rates_4 = pd.Series([mortality_normal_4, mortality_low_4], index=['ТАГ > 1.7', 'ТАГ <= 1.7'])
sns.barplot(x=rates_4.index, y=rates_4.values, palette='coolwarm')

plt.title('Сравнение летальности в зависимости от ТАГ')
for i, v in enumerate(rates_4.values):
    plt.text(i, v, f"{v:.1f}%", ha='center', fontweight='bold')
plt.show()

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
threshold = 1.7

# 2. Фильтруем данные: берем только пациентов БЕЗ метаболического синдрома (0)
df_no_ms = df_raw[df_raw[col_ms] == 0].copy()

# 3. Разделяем эту группу по уровню ТАГ
group_high_tag = df_no_ms[df_no_ms[col_tag] > threshold]
group_low_tag = df_no_ms[df_no_ms[col_tag] <= threshold]

# 4. Считаем количество и проценты
def get_mortality_stats(group):
    total = len(group)
    if total == 0: return 0, 0
    # Считаем количество нулей (умерших)
    deaths = len(group[group[col_target] == 0])
    percent = (deaths / total) * 100
    return total, deaths, percent

total_high, deaths_high, perc_high = get_mortality_stats(group_high_tag)
total_low, deaths_low, perc_low = get_mortality_stats(group_low_tag)

# 5. Вывод результата
print(f"--- АНАЛИЗ ГРУППЫ БЕЗ МЕТАБОЛИЧЕСКОГО СИНДРОМА (N={len(df_no_ms)}) ---")
print(f"\n1. ТАГ выше {threshold}:")
print(f"   - Всего пациентов: {total_high}")
print(f"   - Умерло: {deaths_high}")
print(f"   - Процент летальности: {perc_high:.2f}%")

print(f"\n2. ТАГ ниже или равно {threshold}:")
print(f"   - Всего пациентов: {total_low}")
print(f"   - Умерло: {deaths_low}")
print(f"   - Процент летальности: {perc_low:.2f}%")

# Визуализация для отчета
plt.figure(figsize=(6, 5))
sns.barplot(x=['ТАГ > 1.7', 'ТАГ <= 1.7'], y=[perc_high, perc_low], palette='muted')
plt.title('Летальность пациентов БЕЗ МС в зависимости от ТАГ')
plt.ylabel('Летальность (%)')
plt.show()

In [ ]:
# 1. Задаем параметры
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
threshold = 1.7

# 2. Фильтруем данные: берем только пациентов С метаболическим синдромом (1)
df_yes_ms = df_raw[df_raw[col_ms] == 1].copy()

# 3. Разделяем группу С МС по уровню ТАГ
group_ms_high_tag = df_yes_ms[df_yes_ms[col_tag] > threshold]
group_ms_low_tag = df_yes_ms[df_yes_ms[col_tag] <= threshold]

# 4. Считаем количество и проценты
def get_mortality_stats(group):
    total = len(group)
    if total == 0: return 0, 0, 0
        
    deaths = len(group[group[col_target] == 0])
    percent = (deaths / total) * 100
    return total, deaths, percent

total_h, deaths_h, perc_h = get_mortality_stats(group_ms_high_tag)
total_l, deaths_l, perc_l = get_mortality_stats(group_ms_low_tag)

# 5. Вывод результата
print(f"--- АНАЛИЗ ГРУППЫ С МЕТАБОЛИЧЕСКИМ СИНДРОМОМ (N={len(df_yes_ms)}) ---")
print(f"\n1. ТАГ выше {threshold}:")
print(f"   - Всего пациентов: {total_h}")
print(f"   - Умерло: {deaths_h}")
print(f"   - Процент летальности: {perc_h:.2f}%")

print(f"\n2. ТАГ ниже или равно {threshold}:")
print(f"   - Всего пациентов: {total_l}")
print(f"   - Умерло: {deaths_l}")
print(f"   - Процент летальности: {perc_l:.2f}%")

# Визуализация
plt.figure(figsize=(6, 5))
sns.barplot(x=[f'МС + ТАГ > {threshold}', f'МС + ТАГ <= {threshold}'], y=[perc_h, perc_l], palette='Reds_r')
plt.title('Летальность пациентов С МЕТАБОЛИЧЕСКИМ СИНДРОМОМ')
plt.ylabel('Летальность (%)')
for i, v in enumerate([perc_h, perc_l]):
    plt.text(i, v, f"{v:.2f}%", ha='center', fontweight='bold')
plt.show()

# Нормализация данных

In [ ]:
# 1. Нормализация (MinMax - приведение значений в диапазон от 0 до 1)
minmax_scaler = MinMaxScaler()
X_transf_norm = X.copy()
X_transf_norm[features_list] = minmax_scaler.fit_transform(X[features_list])

# 2. Стандартизация (StandardScaler - среднее 0, стандартное отклонение 1)
stand_scaler = StandardScaler()
X_transf_stand = X.copy()
X_transf_stand[features_list] = stand_scaler.fit_transform(X[features_list])

display(X_transf_stand.head())

# Сохранение обработанной таблицы

In [ ]:
df_without_smote = pd.concat([X_transf_norm, y], axis=1)

df_without_smote.to_csv(OUT / 'test_without_SMOTE.csv', index=False, encoding='utf-8-sig')

# Построение модели Случайного леса

In [ ]:
X_real = X_transf_norm

X_train, X_test, y_train, y_test = train_test_split(
    X_real, y, test_size=0.2, random_state=42, stratify=y
)

forest_balanced = RandomForestClassifier(
    n_estimators=1000, 
    max_depth=5,
    class_weight='balanced', 
    random_state=42
)
forest_balanced.fit(X_train, y_train)

y_pred = forest_balanced.predict(X_test)
print(f"Реальный Recall (чувствительность): {recall_score(y_test, y_pred):.2f}")

# Вывод графика важных факторов в предсказании исхода

In [ ]:
importances = forest_balanced.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Важность признаков (Реальные данные + Веса классов)")
plt.bar(range(X_train.shape[1]), importances[indices], color="firebrick", align="center")
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.show()

# Вывод графика методом SHAP для лучшей интерпритации значимости факторов

In [ ]:
explainer = shap.TreeExplainer(forest_balanced)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    s_vals_death = shap_values[0] 
    base_val_death = explainer.expected_value[0]
else:
    s_vals_death = shap_values[:, :, 0]
    base_val_death = explainer.expected_value[0]

plt.figure()
shap.summary_plot(s_vals_death, X_test, show=False)
plt.title("Влияние факторов на РИСК СМЕРТИ")
plt.show()

# Оцнека точности модели

In [ ]:
# 1. Получаем предсказания
y_pred = forest_balanced.predict(X_test)
y_proba = forest_balanced.predict_proba(X_test) # Вероятности для обоих классов

# 2. Рассчитываем метрики специально для класса 0 (СМЕРТЬ)
acc = accuracy_score(y_test, y_pred)
# Precision и Recall считаем для 0, так как это наше целевое событие для мониторинга
prec_death = precision_score(y_test, y_pred, pos_label=0) 
rec_death = recall_score(y_test, y_pred, pos_label=0)     
f1_death = f1_score(y_test, y_pred, pos_label=0)

# ROC-AUC для риска смерти (это вероятность класса 0)
auc_death = roc_auc_score(y_test, y_proba[:, 0])

print("--- ИТОГОВЫЕ МЕТРИКИ (Целевое событие: СМЕРТЬ [0]) ---")
print(f"Общая точность (Accuracy): {acc:.4f}")
print(f"Чувствительность к смерти (Recall): {rec_death:.4f}")
print(f"Прогностическая ценность смерти (Precision): {prec_death:.4f}")
print(f"F1-мера для класса 'Смерть': {f1_death:.4f}")
print(f"Площадь под ROC-кривой (риск смерти): {auc_death:.4f}")

print("\n--- ДЕТАЛЬНЫЙ ОТЧЕТ ---")
print(classification_report(y_test, y_pred, target_names=['Смерть (0)', 'Жив (1)']))

In [ ]:
# 3. Визуализация матрицы ошибок
plt.figure(figsize=(7, 6))
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Смерть (0)', 'Жив (1)'], 
            yticklabels=['Смерть (0)', 'Жив (1)'])
plt.title('Матрица ошибок (1-жив, 0-смерть)')
plt.xlabel('Предсказано моделью')
plt.ylabel('Реальный исход')
plt.show()

# Генерация искуственных данных

In [ ]:
# 1. Инициализируем SMOTE строго по методологии из отчета:
smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)

# 2. Генерация данных 
# Берем нормализованную матрицу X_transf_norm и целевую переменную y из прошлого шага
X_res_norm, y_res = smote.fit_resample(X_transf_norm, y)

# 3. Объединяем сгенерированные признаки и исходы в новый датафрейм
df_smote = pd.concat([X_res_norm, y_res], axis=1)

# Сохранение таблицы с искуственными данными

In [ ]:
df_smote.to_csv(OUT / 'test_with_SMOTE.csv', index=False, encoding='utf-8-sig')

# Обучение модели на данных вместе с искуственными

In [ ]:
forest_smote = RandomForestClassifier(n_estimators=1000, max_depth=5, random_state=42)
forest_smote.fit(X_res_norm, y_res.values.ravel())

# Вывод графика важных факторов в предсказании исхода

In [ ]:
importances_s = forest_smote.feature_importances_
indices_s = np.argsort(importances_s)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Важность признаков (Модель обучена на SMOTE)")
plt.bar(range(X_res_norm.shape[1]), importances_s[indices_s], color="darkorange", align="center")
plt.xticks(range(X_res_norm.shape[1]), X_res_norm.columns[indices_s], rotation=90)
plt.show()

# Оценка точности

In [ ]:
y_pred_smote = forest_smote.predict(X_test)
y_proba_smote = forest_smote.predict_proba(X_test)

# Метрики точности (Целевое событие: СМЕРТЬ [0])
acc_s = accuracy_score(y_test, y_pred_smote)
rec_s = recall_score(y_test, y_pred_smote, pos_label=0)
prec_s = precision_score(y_test, y_pred_smote, pos_label=0)
auc_s = roc_auc_score(y_test, y_proba_smote[:, 0])

print(f"\nИТОГИ (SMOTE модель на реальном тесте):")
print(f"Общая точность (Accuracy): {acc_s:.4f}")
print(f"Чувствительность к смерти (Recall): {rec_s:.4f}")
print(f"Прогностическая ценность смерти (Precision): {prec_s:.4f}")
print(f"ROC-AUC (Риск смерти): {auc_s:.4f}")

In [ ]:
plt.figure(figsize=(7, 6))
conf_s = confusion_matrix(y_test, y_pred_smote)
sns.heatmap(conf_s, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Смерть (0)', 'Жив (1)'], 
            yticklabels=['Смерть (0)', 'Жив (1)'])
plt.title('Матрица ошибок (SMOTE модель)')
plt.xlabel('Предсказано моделью')
plt.ylabel('Реальный исход')
plt.show()

# Вывод графика методом SHAP

In [ ]:
sns.set_style('whitegrid')

# 1. Инициализация TreeExplainer для обученного случайного леса
explainer = shap.TreeExplainer(forest_smote)

# 2. Вычисляем SHAP-значения для реальных нормализованных данных (X_transf_norm)
# Синтаксис explainer(X) возвращает готовый объект Explanation со всеми базовыми значениями
shap_values_obj = explainer(X_transf_norm)

# 3. Определяем индекс класса "Умер" 
# В вашей колонке 'Исход госпитализации 1 - жив; 0 - умер' смерть обозначена нулем (0)
dead_class_idx = np.where(forest_smote.classes_ == 0)[0][0]

# Извлекаем значения SHAP именно для риска летального исхода
shap_values_dead = shap_values_obj[:, :, dead_class_idx]

In [ ]:
# 1. Bar plot - Средний абсолютный вклад каждого признака
plt.figure()
shap.plots.bar(shap_values_dead, show=False)
plt.savefig(OUT / "shap_bar.png", dpi=700, bbox_inches="tight")
plt.show()

# 2. Beeswarm plot - Детальное распределение влияния
# Показывает, как ВЫСОКИЕ (красные) или НИЗКИЕ (синие) значения признака толкают прогноз
fig = plt.figure()
shap.plots.beeswarm(shap_values_dead, show=False)
plt.savefig(OUT / "shap_beeswarm.png", dpi=700, bbox_inches="tight")
plt.show()

# Находим индексы реальных пациентов, которые умерли (значение 0)
real_dead_indices = np.where(y.values.ravel() == 0)[0]

# Построим графики для первых 3-х реальных умерших пациентов (по аналогии с CRT_3.ipynb)
for i in real_dead_indices[:1]:
    print(f"Разбор факторов для реального пациента №{i} (Исход: Умер):")
    
    plt.figure(figsize=(10, 6))
    # Строим водопадную диаграмму напрямую из объекта Explanation
    shap.plots.waterfall(shap_values_dead[i], show=False)
    
    # Сохраняем индивидуальный график
    plt.savefig(OUT / f"patient_{i}_waterfall.png", dpi=700, bbox_inches="tight")
    plt.show()

# Таблица сопряженности для проверки зависимости категориальных факторов (ФВ и исход)

In [ ]:
col_ef = 'ФВ менее, либо равно 45% 0 - нет; 1 - да'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

ct = pd.crosstab(
    df_raw[col_ef], 
    df_raw[col_target], 
    margins=False
)

ct.index = ['ФВ > 45%', 'ФВ <= 45%']
ct.columns = ['Умерло (0)', 'Выжило (1)']

display(ct)

In [ ]:
odds_ratio, p_value = fisher_exact(ct)

total_normal_ef = ct.loc['ФВ > 45%'].sum()
total_low_ef = ct.loc['ФВ <= 45%'].sum()

mortality_normal = (ct.loc['ФВ > 45%', 'Умерло (0)'] / total_normal_ef) * 100
mortality_low = (ct.loc['ФВ <= 45%', 'Умерло (0)'] / total_low_ef) * 100

print(f"Летальность в группе ФВ > 45%: {mortality_normal:.2f}%")
print(f"Летальность в группе ФВ <= 45%: {mortality_low:.2f}%")

print(f"P-value (Фишер): {p_value:.4f}")
print(f"Отношение шансов (Odds Ratio): {odds_ratio:.2f}")

In [ ]:
plt.figure(figsize=(6, 5))
rates = pd.Series([mortality_normal, mortality_low], index=['ФВ > 45%', 'ФВ <= 45%'])
sns.barplot(x=rates.index, y=rates.values, palette='coolwarm')

plt.title('Сравнение летальности в зависимости от Фракции Выброса')
for i, v in enumerate(rates.values):
    plt.text(i, v, f"{v:.1f}%", ha='center', fontweight='bold')
plt.show()